<a href="https://colab.research.google.com/github/iws3/llm_from_Scratch/blob/main/llm_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from datasets import load_dataset
from tqdm.auto import tqdm

NUM_SAMPLES = 100_000  # Start with 100K samples (~50-100MB text)

print("Loading CulturaX Urdu dataset (streaming)...")
dataset = load_dataset(
    "uonlp/CulturaX",
    "en",                    # Urdu language code
    split="train",
    streaming=True          # Don't download everything
)

# Collect samples
raw_texts = []
for i, sample in enumerate(tqdm(dataset, total=NUM_SAMPLES, desc="Downloading")):
    if i >= NUM_SAMPLES:
        break
    raw_texts.append(sample["text"])

print(f"\nDownloaded {len(raw_texts)} samples")
print(f"Total characters: {sum(len(t) for t in raw_texts):,}")
print(f"\nSample text (first 500 chars):")
print(raw_texts[0][:500])

Loading CulturaX Urdu dataset (streaming)...


Resolving data files:   0%|          | 0/3072 [00:00<?, ?it/s]

Downloading:   0%|          | 0/100000 [00:00<?, ?it/s]


Downloaded 100000 samples
Total characters: 355,512,985

Sample text (first 500 chars):
DOT Announces 2008 Exploration Program - Redorbit
CALGARY, ALBERTA--(Marketwire - July 31, 2008) - DOT Resources Ltd. (TSX VENTURE:DOT) ("DOT" or the "Corporation") is pleased to announce it will be completing a deep 3D Induced Polarization/ Resistivity ("IP") survey on its Dot porphyry copper property (the "Property") located 17 kilometres south of the Highland Valley Mining complex, in central British Columbia. The purpose of the 3D IP survey is to test the depth extension of the copper-molybd


In [11]:
import re
import unicodedata
import regex

def clean_english_text(text: str) -> str:
    """
    Clean a single English text document.

    Steps:
    1. Remove URLs
    2. Remove HTML tags and entities
    3. Remove email addresses
    4. Normalize Unicode (NFKC normalization)
    5. Remove non-English characters (keep English + punctuation + digits)
    6. Normalize repeated punctuation (..., --, etc.)
    7. Normalize whitespace
    """

    # Step 1: Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Step 2: Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove HTML entities
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    text = re.sub(r'&#\d+;', ' ', text)

    # Step 3: Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # Step 4: Unicode normalization (NFKC)
    # This normalizes different representations of the same character
    text = unicodedata.normalize('NFKC', text)

    # Step 5: Keep only English characters, basic punctuation, digits, and whitespace
    english_pattern = regex.compile(
        r'[^'
        r'a-zA-Z'           # English alphabet
        r'0-9'              # Western digits
        r'\s'               # Whitespace
        r'.,:;!?\-"(\']'  # Basic Latin punctuation
    )
    text = english_pattern.sub(' ', text)

    # Step 6: Normalize repeated punctuation
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r'-\s*-+', '-', text)
    text = re.sub(r'-{2,}', '-', text)
    text = re.sub(r',{2,}', ',', text)
    # Remove any lingering Urdu-specific punctuation and normalize spaces around punctuation
    text = re.sub(r'[۔،٪]', ' ', text)
    text = re.sub(r'\s+[\.,:;!?\-"(\']\s+', ' ', text)

    # Step 7: Normalize whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)  # Max 2 newlines
    text = re.sub(r'[^\S\n]+', ' ', text)    # Collapse spaces (but keep newlines)
    text = text.strip()

    return text


def is_mostly_english(text: str, threshold: float = 0.5) -> bool:
    """
    Check if text is mostly English characters.
    This filters out documents that are primarily non-English languages.

    threshold: minimum fraction of characters that must be English
    """
    if len(text) == 0:
        return False
    english_chars = len(regex.findall(r'[a-zA-Z]', text))
    return (english_chars / len(text)) > threshold


# Test the cleaning function
sample = raw_texts[0]
print("=== BEFORE CLEANING ===")
print(sample[:300])
print("\n=== AFTER CLEANING ===")
cleaned = clean_english_text(sample)
print(cleaned[:300])
print(f"\nIs mostly English: {is_mostly_english(cleaned)}")

=== BEFORE CLEANING ===
DOT Announces 2008 Exploration Program - Redorbit
CALGARY, ALBERTA--(Marketwire - July 31, 2008) - DOT Resources Ltd. (TSX VENTURE:DOT) ("DOT" or the "Corporation") is pleased to announce it will be completing a deep 3D Induced Polarization/ Resistivity ("IP") survey on its Dot porphyry copper prope

=== AFTER CLEANING ===
DOT Announces 2008 Exploration Program Redorbit
CALGARY, ALBERTA-(Marketwire July 31, 2008 DOT Resources Ltd. (TSX VENTURE:DOT ("DOT" or the "Corporation" is pleased to announce it will be completing a deep 3D Induced Polarization Resistivity ("IP" survey on its Dot porphyry copper property (the "Pr

Is mostly English: True


In [3]:
VOCAB_SIZE = 32_000  # Number of tokens in our vocabulary
MIN_FREQUENCY = 2    # Token must appear at least twice (filters noise)

# Special tokens - these have reserved IDs
SPECIAL_TOKENS = [
    "<pad>",    # ID 0: padding
    "<unk>",    # ID 1: unknown
    "<bos>",    # ID 2: beginning of sequence
    "<eos>",    # ID 3: end of sequence
    "<sep>",    # ID 4: separator (for chat format)
    "<|user|>",     # ID 5: user turn marker (for chat)
    "<|assistant|>", # ID 6: assistant turn marker (for chat)
    "<|system|>",    # ID 7: system prompt marker (for chat)
]

In [12]:
print('Cleaning and saving corpus...')

# Create the directory if it doesn't exist
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

with open(CORPUS_FILE, "w", encoding="utf-8") as f:
    for text in raw_texts:
        cleaned_text = clean_english_text(text)
        if is_mostly_english(cleaned_text):
            f.write(cleaned_text + "\n")

print(f"Corpus saved to {CORPUS_FILE}")

Cleaning and saving corpus...
Corpus saved to /data/cleaned/english_corpus.txt


In [13]:
import os
from pathlib import Path
from tokenizers import (
    Tokenizer,
    models,
    trainers,
    pre_tokenizers,
    decoders,
    processors,
    normalizers,
)

PROJECT_ROOT = Path(".").resolve().parent
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
TOKENIZER_DIR = PROJECT_ROOT / "tokenizer" / "english_tokenizer"
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_FILE = str(CLEANED_DIR / "english_corpus.txt")
print(f"Corpus file: {CORPUS_FILE}")
print(f"Tokenizer output: {TOKENIZER_DIR}")

# Verify corpus exists
assert os.path.exists(CORPUS_FILE), f"Corpus not found at {CORPUS_FILE}. Run notebook 01 first!"
file_size_mb = os.path.getsize(CORPUS_FILE) / 1024 / 1024
print(f"Corpus size: {file_size_mb:.1f} MB")

Corpus file: /data/cleaned/english_corpus.txt
Tokenizer output: /tokenizer/english_tokenizer
Corpus size: 334.6 MB
